In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_7.pickle

trying: ['meta_info', 'filename']
me:  1
me:  1
trying: ['meta_info', 'filename']
me:  1
me:  1
trying: ['filenames']
me:  1
trying: ['col_name_corrections']
me:  1
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['factor']
me:  1
trying: ['orig_output']
me:  16
trying: ['Mobile_df']
me:  1
trying: ['pd']
me:  0
trying: ['np']
me:  0
trying: ['Path']
me:  0
trying: ['unique_countries_mobile']
me:  5
trying: ['Broadband_df']
me:  1
trying: ['benchmark_name']
me:  0
trying: ['re']
me:  0
trying: ['dirname']
me:  0
trying: ['data']
me:  1
trying: ['unique_countries_broadband']
me:  3
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable pd
Declaring variable np
Declaring variable Path
Declaring variable benchmark_name
Declaring variable re
Declaring variable dirname
Declaring variable meta_info
Declaring variable filename
Declaring variable meta_info
Declaring variable filename
Declaring variable filenames
Declaring variable col_name_corrections
Declaring variable factor
Declaring vari

In [4]:
%%RecordEvent
%%time
### cell 8 ###

### cell 8 optimized
cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile stats: do first and last in one groupby.agg
mobile_agg = Mobile_df.groupby('Name')[cols].agg(['first','last'])
# split out first/last frames
first_m = mobile_agg.xs('first', level=1, axis=1)
last_m  = mobile_agg.xs('last',  level=1, axis=1)
Mobile_Stats = (
    (last_m['Avg. Avg D Kbps'] - first_m['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_m['Avg. Avg U Kbps'] - first_m['Avg. Avg U Kbps']),
        Change_Latency=(last_m['Avg Lat Ms']      - first_m['Avg Lat Ms'])
    )
)

# Broadband stats: same pattern
broad_agg = Broadband_df.groupby('Name')[cols].agg(['first','last'])
first_b = broad_agg.xs('first', level=1, axis=1)
last_b  = broad_agg.xs('last',  level=1, axis=1)
Broadband_Stats = (
    (last_b['Avg. Avg D Kbps'] - first_b['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_b['Avg. Avg U Kbps'] - first_b['Avg. Avg U Kbps']),
        Change_Latency=(last_b['Avg Lat Ms']      - first_b['Avg Lat Ms'])
    )
)

# combine only the download‐change columns
Total_Stats = (
    Mobile_Stats['Change_Download'].to_frame('Mobile')
    .join(Broadband_Stats['Change_Download'].to_frame('Broadband'))
)

CPU times: user 101 ms, sys: 50.1 ms, total: 151 ms
Wall time: 139 ms


In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_1.pickle

migration speed (bps): 854591775.6726543
---------------------------
variables to migrate:
dirname 141
factor 28
filename 80
Total_Stats 7156
Mobile_df 5474844
Broadband_Stats 9323
pd 72
broad_agg 29345
col_name_corrections 144
orig_output 0
unique_countries_mobile 26036
Broadband_df 2866634
mobile_agg 28651
last_b 9323
unique_countries_broadband 26675
Mobile_Stats 9044
filenames 248
np 72
cols 88
first_b 9323
data 24651
Path 904
first_m 9044
benchmark_name 63
meta_info 80
last_m 9044
re 72
BENCHMARKS_TO_PATHS 2272
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['filenames', 'benchmark_name', 'BENCHMARKS_TO_PATHS', 'Path', 'dirname', 'filename']
Active variables ['Broadband_df', 'Mobile_df']
Intermediate variables ['filenames', 'factor', 'meta_info', 'filename', 'data']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['Broadband_df']
Active variables ['unique_countries_broadband']
Intermediate variables []
Future variables ['Broadband_df', 'Mobile_df']
Modified dataframes
======= Cell 2 =======
Input variables ['Mobile_df']
Active variables ['unique_countries_mobile']
Intermediate variables []
Future variables ['Broadband_df', 'unique_countries_broadband', 'Mobile_df']
Modified dataframes
======= Cell 3 =======
Input variables ['Mobile_df']
Active variables []
Intermediate variables []
Future variables ['Broadband_df', 'unique_countries_broadband', 'Mobile_df', 'unique_countries_mobile']
Modified dataframes
======= Cell 4 =======
Input variables ['Broadband_df']
Active vari

In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/opt_cell_exec_info_8_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[8], f)


In [8]:
opt_output = Out.get(4)

In [9]:
Broadband_Stats_opt = Broadband_Stats
Mobile_Stats_opt = Mobile_Stats
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_8.pickle
assert compare_df(Broadband_Stats_opt, Broadband_Stats)
assert compare_df(Mobile_Stats_opt, Mobile_Stats)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['pd']
me:  0
trying: ['col_name_corrections']
me:  1
trying: ['orig_output']
me:  18
trying: ['Mobile_Stats']
me:  17
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['unique_countries_mobile']
me:  5
trying: ['Broadband_Stats']
me:  17
trying: ['np']
me:  0
trying: ['Broadband_df']
me:  1
trying: ['filenames']
me:  1
trying: ['unique_countries_broadband']
me:  3
trying: ['Path']
me:  0
trying: ['data']
me:  1
trying: ['benchmark_name']
me:  0
trying: ['factor']
me:  1
trying: ['re']
me:  0
trying: ['Total_Stats']
me:  18
trying: ['Mobile_df']
me:  1
trying: ['dirname']
me:  0
Declaring variable pd
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable np
Declaring variable Path
Declaring variable benchmark_name
Declaring variable re
Declaring variable dirname
Declaring variable filename
Declaring variable meta_info
Declaring variable filename
Declaring variable meta_info
Declarin